<hr />

### Football Performance & Team Success

<hr />

<p>Data Visualization</p>

<p>Prof. Maryam Abbasi</p>

<p>Escola Superior de Gestão e Tecnologia - Instituto Politécnico de Santarém</p>

<hr />

<p>Isaac Mendes - 250001162</p>

<p>Rodrigo Calado - 250001513</p>

<p>Tiago Amorim - 120118010</p>

<hr />

#### Import Libraries

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
RANDOM_STATE = 42

print('All libraries imported successfully!')
print(f'   numpy  : {np.__version__}')
print(f'   pandas : {pd.__version__}')

All libraries imported successfully!
   numpy  : 2.3.5
   pandas : 2.3.3


<hr />

#### Dataset Loading and Data Preparation

In [2]:
## Load dataset
path_fpts_data = 'data/understat_per_game.csv'
fpts_df = pd.read_csv(path_fpts_data)

## FIX: Parse date column immediately after loading
fpts_df['date'] = pd.to_datetime(fpts_df['date'])
fpts_df['month'] = fpts_df['date'].dt.month
fpts_df['season_half'] = (fpts_df['date'].dt.month > 6).astype(int)

## FIX: Replace the 10 zero values in ppda_coef with the column median
## (a ppda of exactly 0 is physically impossible and likely a data error)
ppda_median = fpts_df.loc[fpts_df['ppda_coef'] > 0, 'ppda_coef'].median()
fpts_df['ppda_coef'] = fpts_df['ppda_coef'].replace(0, ppda_median)
fpts_df['oppda_coef'] = fpts_df['oppda_coef'].replace(0, fpts_df.loc[fpts_df['oppda_coef'] > 0, 'oppda_coef'].median())

print(f'Dataset loaded: {fpts_df.shape[0]:,} rows x {fpts_df.shape[1]} columns')
print(f'Seasons: {sorted(fpts_df["year"].unique())}')
print(f'Leagues: {list(fpts_df["league"].unique())}')
print(f'Teams: {fpts_df["team"].nunique()}')
print(f'Date range: {fpts_df["date"].min().date()} → {fpts_df["date"].max().date()}')

Dataset loaded: 24,580 rows x 31 columns
Seasons: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Leagues: ['Bundesliga', 'EPL', 'La_liga', 'Ligue_1', 'RFPL', 'Serie_A']
Teams: 168
Date range: 2014-08-01 → 2020-08-01


- The dataset contains 24,580 observations and 31 features.
- It covers football seasons from 2014 to 2020, across six major European leagues: Bundesliga, EPL, La Liga, Ligue 1, RFPL, and Serie A.
- In total, the dataset includes 168 unique teams, providing a comprehensive view of team performance over multiple seasons.

<hr />

#### EDA

In [3]:
## FIX: Exclude 'year' (not a meaningful predictor), and exclude 'wins','draws','loses'
## because they are direct decompositions of 'pts' — including them causes data leakage.
LEAKAGE_COLS = ['year', 'wins', 'draws', 'loses']
RAW_COLS     = ['ppda_att', 'ppda_def', 'oppda_att', 'oppda_def']  # raw counts behind the coefficients

numerical_cols   = [c for c in fpts_df.select_dtypes(include=[np.number]).columns
                    if c not in LEAKAGE_COLS + RAW_COLS]
categorical_cols = fpts_df.select_dtypes(include=['object']).columns.tolist()
# Remove 'date' from categorical since it's now datetime
categorical_cols = [c for c in categorical_cols if c != 'date']

target_col = 'pts'
corr_matrix = fpts_df[numerical_cols].corr()

## General overview
print('=== Dataset Overview ===')
print(fpts_df.shape)
print(fpts_df.info())
print('=' * 40)

## Numerical statistics
print('Numerical statistics:')
print(fpts_df[numerical_cols].describe().T.round(3))
print('=' * 40)

## Categorical statistics
print('Categorical statistics:')
print(fpts_df[categorical_cols].describe().T)
print('=' * 40)

## Missing values check
missing = fpts_df.isnull().sum().sort_values(ascending=False)
missing_plot = missing[missing > 0]

if len(missing_plot) > 0:
    fig_miss = go.Figure(data=[
        go.Bar(x=missing_plot.index, y=missing_plot.values,
               marker_color='indianred', text=missing_plot.values,
               textposition='auto')
    ])
    fig_miss.update_layout(
        title='Missing Values per Column',
        xaxis_title='Column', yaxis_title='Count',
        xaxis_tickangle=45, height=400, showlegend=False
    )
    fig_miss.show()
else:
    print('No missing values found — dataset is complete.')
print('=' * 40)

=== Dataset Overview ===
(24580, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24580 entries, 0 to 24579
Data columns (total 31 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   league        24580 non-null  object        
 1   year          24580 non-null  int64         
 2   h_a           24580 non-null  object        
 3   xG            24580 non-null  float64       
 4   xGA           24580 non-null  float64       
 5   npxG          24580 non-null  float64       
 6   npxGA         24580 non-null  float64       
 7   deep          24580 non-null  int64         
 8   deep_allowed  24580 non-null  int64         
 9   scored        24580 non-null  int64         
 10  missed        24580 non-null  int64         
 11  xpts          24580 non-null  float64       
 12  result        24580 non-null  object        
 13  date          24580 non-null  datetime64[ns]
 14  wins          24580 non-null  int64         
 15 

- The numerical statistics provide an overview of the distribution of key performance metrics such as xG, xGA, and points. These variables will be particularly important for understanding team performance and predicting success.
- Regarding categorical variables, the dataset includes six leagues, with the EPL being the most represented. The most frequent team (Real Sociedad) reflects data availability rather than performance.
- Additionally, wins are the most common match outcome, which aligns with the structure of the dataset.

<hr />

<h3>Do contextual factors such as league or match location influence team success?</h3>

In [ ]:
## Categorical features vs target
print('## Categorical Features vs Target')
top_cat_cols = [c for c in categorical_cols if c in ['league', 'h_a', 'result']]  # only meaningful ones

fig_cat = make_subplots(
    rows=1, cols=len(top_cat_cols),
    subplot_titles=[f'{col} (avg {target_col})' for col in top_cat_cols],
    specs=[[{'type': 'bar'}] * len(top_cat_cols)]
)

for i, col in enumerate(top_cat_cols):
    tab = fpts_df.groupby(col)[target_col].agg(['count', 'mean']).round(2)
    tab = tab.sort_values('mean', ascending=False)

    print(f'\nColumn: {col}')
    print(tab)

    fig_cat.add_trace(
        go.Bar(
            x=tab.index, y=tab['mean'],
            text=tab['mean'].round(2), textposition='auto',
            marker_color=px.colors.sequential.Viridis[:len(tab)],
            showlegend=False
        ),
        row=1, col=i + 1
    )
    fig_cat.update_xaxes(tickangle=45, row=1, col=i + 1)

fig_cat.update_layout(
    height=450,
    title_text='Categorical Features vs Target (Average Points)'
)
fig_cat.show()

## Categorical Features vs Target

Column: league
            count  mean
league                 
Bundesliga   3672  1.38
EPL          4560  1.38
La_liga      4560  1.37
Ligue_1      4358  1.37
Serie_A      4550  1.37
RFPL         2880  1.36

Column: h_a
     count  mean
h_a             
h    12290  1.60
a    12290  1.15

Column: result
        count  mean
result             
w        9189   3.0
d        6202   1.0
l        9189   0.0


- Average points per match are very similar across leagues, suggesting that league context does not significantly influence performance.
- However, teams playing at home achieve higher average points than when playing away, indicating a clear home advantage.
- As expected, wins correspond to 3 points, draws to 1 point, and losses to 0 points. This confirms that the target variable is directly derived from match outcomes, meaning that predicting outcomes is equivalent to predicting points.

<hr />

<h3>How do performance metrics relate to match outcomes?</h3>

In [5]:
## FIX: Correct top_num_cols — nlargest(5) then drop target gives exactly 4 clean features
top_num_cols = corr_matrix[target_col].abs().nlargest(5).index.tolist()
top_num_cols = [c for c in top_num_cols if c != target_col]  # exactly 4 features

print('## Numerical Features vs Target')
print(f'Top correlated features: {top_num_cols}')

## FIX: Correct row/col assignment — was computing 'row' then ignoring it
fig_num = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{col} vs {target_col}' for col in top_num_cols],
    specs=[
        [{'type': 'box'},    {'type': 'box'}],
        [{'type': 'violin'}, {'type': 'violin'}]
    ]
)

colors = px.colors.qualitative.Set2

for i, col in enumerate(top_num_cols):
    ## FIX: Properly derive row and column position from index
    subplot_row = (i // 2) + 1   # features 0,1 → row 1 (box);  features 2,3 → row 2 (violin)
    subplot_col = (i % 2) + 1    # features 0,2 → col 1;        features 1,3 → col 2

    is_box_row = subplot_row == 1

    print(f'\nColumn: {col}')
    grouped = fpts_df.groupby(target_col)[col].agg(['mean', 'std']).round(2)
    print(grouped)

    if is_box_row:
        fig_num.add_trace(
            go.Box(
                y=fpts_df[col], x=fpts_df[target_col].astype(str),
                name=col, marker_color=colors[i],
                boxpoints='outliers'
            ),
            row=subplot_row, col=subplot_col
        )
    else:
        fig_num.add_trace(
            go.Violin(
                y=fpts_df[col], x=fpts_df[target_col].astype(str),
                name=col, marker_color=colors[i],
                box_visible=True, points='outliers'
            ),
            row=subplot_row, col=subplot_col
        )

fig_num.update_layout(
    height=700,
    title_text='Top Numerical Features vs Target — Boxplot & Violin',
    showlegend=False
)
fig_num.update_xaxes(tickangle=45)
fig_num.show()

## Numerical Features vs Target
Top correlated features: ['xpts_diff', 'scored', 'missed', 'xpts']

Column: xpts_diff
     mean   std
pts            
0    0.81  0.61
1    0.36  0.68
3   -1.04  0.68

Column: scored
     mean   std
pts            
0    0.54  0.69
1    0.98  0.82
3    2.38  1.19

Column: missed
     mean   std
pts            
0    2.38  1.19
1    0.98  0.82
3    0.54  0.69

Column: xpts
     mean   std
pts            
0    0.81  0.61
1    1.36  0.68
3    1.96  0.68


- Winning teams tend to outperform expectations (positive xpts_diff), while losing teams underperform relative to expected points.
- Teams that score more goals are significantly more likely to win, whereas low-scoring performances are associated with draws and losses.
- Similarly, teams that concede more goals are more likely to lose, highlighting the importance of defensive performance.
- Finally, expected points (xpts) show a strong alignment with actual results, indicating that advanced metrics are effective predictors of match outcomes.

<hr />

#### Correlation Analysis & Key Distributions

In [6]:
## Full correlation matrix (clean — year and leakage cols excluded)
print(corr_matrix[target_col].sort_values(ascending=False).round(4))

fig_heatmap = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect='auto',
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    title=f'Correlation Matrix — Target: {target_col}'
)
fig_heatmap.update_layout(height=700, font_size=10)
fig_heatmap.show()

strong_corr = corr_matrix[target_col].abs().nlargest(6)
print(f'\nTop 5 correlations with {target_col}:')
for col, corr in list(strong_corr.items())[1:]:  # skip pts itself
    print(f'  {col}: {corr:.4f}')
print('=' * 40)

pts             1.0000
scored          0.6572
xpts            0.6010
npxGD           0.5567
xG              0.4684
npxG            0.4503
xGA_diff        0.4029
deep            0.2182
oppda_coef      0.1204
month          -0.0014
season_half    -0.0019
ppda_coef      -0.1053
deep_allowed   -0.2049
npxGA          -0.4162
xGA            -0.4341
xG_diff        -0.4454
missed         -0.6015
xpts_diff      -0.7798
Name: pts, dtype: float64



Top 5 correlations with pts:
  xpts_diff: 0.7798
  scored: 0.6572
  missed: 0.6015
  xpts: 0.6010
  npxGD: 0.5567


<hr />

<h3>What is the distribution of key performance metrics?</h3>

In [ ]:
## Key metrics distributions: xG, xGA, pts
print('## Key Metrics Distributions')
key_cols = ['xG', 'xGA', 'pts']
key_cols = [c for c in key_cols if c in fpts_df.columns]
n_cols = len(key_cols)

fig_dist = make_subplots(
    rows=2, cols=n_cols,
    subplot_titles=key_cols,
    specs=[[{'type': 'histogram'}] * n_cols,
           [{'type': 'box'}] * n_cols],
    vertical_spacing=0.12
)

colors = px.colors.qualitative.Set3

for i, col in enumerate(key_cols):
    col_pos = i + 1
    mean_val = fpts_df[col].mean()

    fig_dist.add_trace(
        go.Histogram(
            x=fpts_df[col], name=col,
            nbinsx=25, marker_color=colors[i], opacity=0.8
        ),
        row=1, col=col_pos
    )
    fig_dist.add_trace(
        go.Box(
            y=fpts_df[col], name=f'{col}_box',
            marker_color=colors[i], showlegend=False
        ),
        row=2, col=col_pos
    )
    fig_dist.add_vline(
        x=mean_val, line_dash='dash', line_color='red',
        annotation_text=f'μ={mean_val:.2f}',
        row=1, col=col_pos
    )

fig_dist.update_layout(
    height=600,
    title_text='Distribution + Boxplot of Key Metrics',
    showlegend=False
)
fig_dist.show()

- The distributions of xG and xGA are very similar, suggesting that teams score and concede comparable expected goals on average.
- Both variables are right-skewed, indicating that most matches have relatively low expected goals, with a small number of high-scoring outliers.
- The distribution of points is discrete, reflecting the structure of football scoring (win, draw, loss). This highlights that predicting points is closely related to predicting match outcomes.

<hr />

<h3>Which variables are most strongly correlated with team success?</h3>

In [ ]:
## Top features focused heatmap
print('## Top Features Correlation')
top_features_corr = fpts_df[top_num_cols + [target_col]].corr()
fig_heatmap_top = px.imshow(
    top_features_corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    title='Top Features Correlation with Target'
)
fig_heatmap_top.show()

- The heatmap reveals a strong positive correlation between points and goals scored, and a strong negative correlation with goals conceded. This indicates that both offensive and defensive performance are key drivers of success.
- Additionally, expected points (xpts) show a strong positive correlation with actual points, suggesting that advanced performance metrics are effective predictors of match outcomes.
- Finally, xpts_diff is strongly negatively correlated with points, indicating that teams performing below expectations tend to achieve worse results, while outperforming expectations is associated with better performance.

<hr />

#### Team-Season Aggregation

Aggregating match-level data to **season-level** per club. This is the unit of analysis needed to compare club performance across seasons — answering the core research question.

> Which teams have been the most consistently successful over time?

In [ ]:
## Build team-season summary
team_season = fpts_df.groupby(['team', 'year', 'league']).agg(
    total_pts      = ('pts',       'sum'),
    avg_xG         = ('xG',        'mean'),
    avg_xGA        = ('xGA',       'mean'),
    avg_npxG       = ('npxG',      'mean'),
    avg_ppda       = ('ppda_coef', 'mean'),
    avg_oppda      = ('oppda_coef','mean'),
    avg_deep       = ('deep',      'mean'),
    total_wins     = ('wins',      'sum'),
    total_draws    = ('draws',     'sum'),
    total_losses   = ('loses',     'sum'),
    games_played   = ('pts',       'count')
).reset_index()

team_season['win_rate']  = team_season['total_wins']   / team_season['games_played']
team_season['xG_diff']   = team_season['avg_xG']       - team_season['avg_xGA']
team_season['pts_per_game'] = team_season['total_pts'] / team_season['games_played']

print(f'Team-season table: {team_season.shape[0]} rows ({team_season["team"].nunique()} teams × {team_season["year"].nunique()} seasons)')
print(team_season.sort_values('total_pts', ascending=False).head(10).to_string(index=False))

## Top clubs by average pts per season
top_clubs = (
    team_season.groupby('team')['total_pts']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

fig_top = px.bar(
    x=top_clubs.index, y=top_clubs.values,
    labels={'x': 'Team', 'y': 'Avg Points per Season'},
    title='Top 15 Clubs by Average Season Points (2014–2019)',
    color=top_clubs.values,
    color_continuous_scale='Viridis'
)
fig_top.update_layout(xaxis_tickangle=45, showlegend=False, coloraxis_showscale=False)
fig_top.show()

- This plot shows the top 15 clubs based on their average points per season between 2014 and 2019.
- Barcelona has the highest average points, followed closely by Juventus and Paris Saint-Germain, indicating strong and consistent performance over the analyzed period.

<hr />
<h3>Does offensive performance (xG) translate into team success across different leagues?</h3>

In [ ]:
## xG vs total points scatter (season level)
fig_scatter = px.scatter(
    team_season,
    x='avg_xG', y='total_pts',
    color='league',
    hover_name='team',
    hover_data={'year': True, 'win_rate': ':.2f'},
    trendline='ols',
    title='Average xG vs Season Points — by League',
    labels={'avg_xG': 'Avg xG per game', 'total_pts': 'Total season points'}
)
fig_scatter.update_layout(height=550)
fig_scatter.show()

- This plot shows a clear positive relationship between average xG per game and total season points, indicating that teams creating more scoring opportunities tend to achieve better results.
- This pattern is consistent across all leagues, suggesting that offensive performance is a strong and reliable predictor of success, regardless of league context.

<hr />

#### Machine Learning — Predicting Match Outcome

**Target**: binary classification — did the team earn points (win or draw = 1, loss = 0)?  
**Features**: performance metrics from the actual football dataset (xG, xGA, ppda, etc.)  
**Models**: Logistic Regression and Random Forest, both trained on `fpts_df`.

In [9]:
## FIX: ML now uses the actual football dataset, not synthetic make_classification data.
## FIX: Features exclude leakage columns (wins/draws/loses are decompositions of pts).

## NOTE: Exclude 'scored','missed','xpts','xpts_diff','xG_diff','xGA_diff'
## — they are derived from the final score and cause data leakage.
## Keep only play-quality metrics available independently of the result.
ML_FEATURES = ['xG', 'xGA', 'npxG', 'npxGA', 'deep', 'deep_allowed',
               'npxGD', 'ppda_coef', 'oppda_coef']

## Binary target: 1 = earned points (win or draw), 0 = loss
X = fpts_df[ML_FEATURES].copy()
y = (fpts_df['pts'] >= 1).astype(int)

print(f'Features: {ML_FEATURES}')
print(f'Target class balance — earned points: {y.sum():,} ({y.mean():.1%}) | loss: {(1-y).sum():,} ({(1-y).mean():.1%})')

## Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

## Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train set: {X_train.shape[0]:,} rows | Test set: {X_test.shape[0]:,} rows')

Features: ['xG', 'xGA', 'npxG', 'npxGA', 'deep', 'deep_allowed', 'npxGD', 'ppda_coef', 'oppda_coef']
Target class balance — earned points: 15,391 (62.6%) | loss: 9,189 (37.4%)
Train set: 19,664 rows | Test set: 4,916 rows


In [10]:
## Train models
lr  = LogisticRegression(random_state=RANDOM_STATE, max_iter=500)
rf  = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)

lr.fit(X_train_sc, y_train)
rf.fit(X_train, y_train)          # Random Forest does not need scaling

lr_preds  = lr.predict(X_test_sc)
rf_preds  = rf.predict(X_test)
lr_proba  = lr.predict_proba(X_test_sc)[:, 1]
rf_proba  = rf.predict_proba(X_test)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, lr_preds, target_names=['Loss', 'Points']))

print('=== Random Forest ===')
print(classification_report(y_test, rf_preds, target_names=['Loss', 'Points']))

=== Logistic Regression ===
              precision    recall  f1-score   support

        Loss       0.71      0.59      0.64      1838
      Points       0.78      0.85      0.81      3078

    accuracy                           0.75      4916
   macro avg       0.74      0.72      0.73      4916
weighted avg       0.75      0.75      0.75      4916

=== Random Forest ===
              precision    recall  f1-score   support

        Loss       0.69      0.60      0.64      1838
      Points       0.78      0.84      0.81      3078

    accuracy                           0.75      4916
   macro avg       0.73      0.72      0.72      4916
weighted avg       0.75      0.75      0.75      4916



<hr />
<h3>How well can we predict match outcomes using performance metrics?</h3>

In [ ]:
## Confusion matrices side by side
fig_cm = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Logistic Regression', 'Random Forest'],
    specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}]]
)

for col_pos, (preds, label) in enumerate([(lr_preds, 'LR'), (rf_preds, 'RF')], start=1):
    cm = confusion_matrix(y_test, preds)
    fig_cm.add_trace(
        go.Heatmap(
            z=cm, x=['Loss', 'Points'], y=['Loss', 'Points'],
            text=cm, texttemplate='%{text}',
            colorscale='Blues', showscale=False
        ),
        row=1, col=col_pos
    )

fig_cm.update_layout(height=400, title_text='Confusion Matrices — Test Set')
fig_cm.show()

- Both models achieve similar performance, with Logistic Regression showing slightly fewer misclassifications overall compared to Random Forest.
- In particular, Logistic Regression correctly identifies more instances of the "Points" class, while also making slightly fewer errors when predicting losses.
- However, both models tend to misclassify losses as points, indicating that predicting negative outcomes remains more challenging.
- Overall, these results suggest that performance metrics are useful predictors of match outcomes, although there is still room for improvement.

<hr />
<h3>How well do the models distinguish between match outcomes?</h3>

In [ ]:
## ROC curves
fig_roc = go.Figure()
for proba, name, color in [(lr_proba, 'Logistic Regression', '#636EFA'),
                            (rf_proba, 'Random Forest',       '#EF553B')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc     = auc(fpr, tpr)
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines', name=f'{name} (AUC={roc_auc:.3f})',
        line=dict(color=color, width=2)
    ))

fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    line=dict(dash='dash', color='gray'), name='Random baseline'
))
fig_roc.update_layout(
    title='ROC Curve — Match Outcome Prediction',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=500
)
fig_roc.show()

- The Logistic Regression model achieves a slightly higher AUC (0.826) compared to the Random Forest (0.815), indicating a marginally better performance.
- However, both models perform very similarly and clearly outperform the random baseline, demonstrating that they are effective at distinguishing between different match outcomes.
- This suggests that the selected performance metrics provide strong predictive power.

<hr />
<h3>Which performance metrics are the most important for predicting team success?</h3>

In [12]:
## Feature importance — Random Forest
importance_df = pd.DataFrame({
    'feature':   ML_FEATURES,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

fig_fi = go.Figure(go.Bar(
    x=importance_df['importance'],
    y=importance_df['feature'],
    orientation='h',
    marker_color=px.colors.sequential.Viridis_r[:len(importance_df)]
))
fig_fi.update_layout(
    title='Feature Importance — Random Forest',
    xaxis_title='Importance',
    yaxis_title='Feature',
    height=450
)
fig_fi.show()

print('\nTop predictors of match outcome:')
for _, row in importance_df.sort_values('importance', ascending=False).head(5).iterrows():
    print(f'  {row["feature"]}: {row["importance"]:.4f}')


Top predictors of match outcome:
  npxGD: 0.1897
  xGA: 0.1730
  xG: 0.1229
  npxGA: 0.1174
  oppda_coef: 0.1007


- The Random Forest model identifies npxGD (Non-Penalty Expected Goal Difference) as the most important feature, followed by xGA, xG, and npxGA.
- This suggests that the difference between attacking and defensive performance is the strongest predictor of success, rather than individual metrics alone.
- Additionally, both offensive (xG) and defensive (xGA) metrics rank highly, indicating that success is driven by a combination of scoring ability and defensive stability.